# AXI test 6: requests in ordinary words (Gemini 3.5 Flash Lite, free tier)
Until now every request named its targets by coordinate ("Delete the service at (2,1)"). Here the request is written in words ("Remove the billing service."), and a new piece of code reads the words against a table of service names. 140 requests are the same graphs and the same correct changes as before; 20 more requests are deliberately ambiguous ("Remove the billing service." when "billing api" and "billing worker" both exist), where the correct reaction is to change nothing. The prediction and the reading rules were written before this run in `docs/WORDS_TEST.md`.

**What to know**
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU). Nothing is downloaded.
* **A daily limit.** The free key allows about 500 calls a day. This test needs 320 calls (160 + 160; the first 6 of them are the check), so it should finish in one day. If it stops, everything finished is saved on your Google Drive: run **Cells 2 to 5 again the next day**, then the cell that stopped.
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in any file. Never paste it in the chat.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on (it is per notebook, so do it again for this one).
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-words.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_words")

### Cell 5: connect to the model and check it with 6 requests (about 1 minute). It prints SMOKE OK or STOP. If it says STOP, do not run the next cells: send me the printout

In [ ]:
import json, time
from google.colab import userdata
from axi.experiments.api_llm import QuotaStop, ServiceBusy, build_gemini_llm
from axi.experiments.word_test import word_set, run_word_regime, print_words, print_verdict, verdict_words, verdict_words_format
from axi.experiments.bigger_model import mean_seconds
from axi.experiments.api_test import smoke_report

try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
WANTED = "gemini-3.5-flash-lite"
CACHE = "axi_words_cache.jsonl"
print("reply file:", store.restore(CACHE))
llm, MODEL = build_gemini_llm(KEY, WANTED, store.path(CACHE), daily_limit=480, min_interval=4.5, on_write=lambda: store.persist(CACHE))
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run the cell that stopped again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once (if this page is still open, you do not need the earlier cells). Do NOT run it again and again: Google counts failed tries against the free daily limit."
T0 = time.time()
def prog(i, n): print(f"  {i}/{n} done   {time.time() - T0:6.0f} s   calls today: {llm.calls_today()}" + ("" if store.persistent else "   (NOT saved to Drive)"), end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None
res_rule = res_norule = None
tasks = word_set()
smoke = go(lambda: run_word_regime(llm, tasks[:6], True, "/content/smoke_words.jsonl", meta=dict(model=MODEL, smoke=True)))
if smoke:
    for r, t in zip(smoke["rows"], tasks): print(f"{r['kind']:14s} {r['seconds']:5.1f} s  {t.instruction[:60]!r}  ->  {r['text'][:80]!r}")
    print("totals so far:", llm.usage())
    ok, msg = smoke_report(smoke["rows"]); print("\n" + msg)

### Cell 6: rule stated in the prompt. 160 requests, one at a time (about 15 to 30 minutes; the first 6 are already done)

In [ ]:
T0 = time.time()
name = "axi_words_rule"
print("progress file:", store.restore(name + ".jsonl"))
res_rule = go(lambda: run_word_regime(llm, tasks, True, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=7), progress=store.progress(name + ".jsonl", prog)))
if res_rule:
    print(); print_words(res_rule["summary"], "rule stated, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_rule["rows"]), 2), "  calls today:", llm.calls_today())
    files.download(save(res_rule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 7: no rule in the prompt. 160 requests (about 15 to 30 minutes). Prints all the reading-rule lines at the end

In [ ]:
assert res_rule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = "axi_words_norule"
print("progress file:", store.restore(name + ".jsonl"))
res_norule = go(lambda: run_word_regime(llm, tasks, False, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=7), progress=store.progress(name + ".jsonl", prog)))
if res_norule:
    print(); print_words(res_norule["summary"], "no rule, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_norule["rows"]), 2), "  calls today:", llm.calls_today())
    print_verdict(verdict_words({True: res_rule["summary"], False: res_norule["summary"]}) + verdict_words_format({True: res_rule["rows"], False: res_norule["rows"]}), "both regimes")
    print("totals:", llm.usage())
    files.download(save(res_norule, store.path(name + ".json"))); store.persist(name + ".json")
    files.download(store.path(CACHE))

### Done
Files downloaded: `axi_words_rule.json`, `axi_words_norule.json` and `axi_words_cache.jsonl`. Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.